In [1]:
from typing import TypeAlias

import numpy as np
from numpy.typing import NDArray

Tensor: TypeAlias = NDArray[np.float32]

In [2]:
batch_size = 64
d_in = 12
d_h = 24
d_out = 8

rng = np.random.default_rng(0)
input = rng.normal(size=(batch_size, d_in)).astype(np.float32)
W_ih = rng.normal(size=(d_h, d_in)).astype(np.float32)
W_ho = rng.normal(size=(d_out, d_h)).astype(np.float32)
output = input @ W_ih.T @ W_ho.T

In [3]:
rng = np.random.default_rng(0)

class Util:
    @staticmethod
    def generate_random_tensor(size: int | tuple[int, ...]) -> Tensor:
        return rng.normal(size=size).astype(np.float32)

class LinearLayer:
    def __init__(self, d_in: int, d_out: int, has_bias: bool = True):
        self.W = Util.generate_random_tensor((d_out, d_in))
        if has_bias:
            self.bias = Util.generate_random_tensor(d_out)
        else:
            self.bias = None

    def forward(self, x: Tensor) -> Tensor:
        retval = input @ self.W.T
        if self.bias is not None:
            retval += self.bias
        return retval

In [4]:
from math import ceil 
from typing import Literal

image = np.arange(7 * 7).reshape(7, -1)
filter = np.arange(9).reshape(3, -1)

i_size = image.shape[0]
f_size = filter.shape[0]
stride = 1
pad: Literal["same"] | int = "same"
if pad == "same":
    pad = max(0, (ceil(i_size / stride) - 1) * stride + f_size - i_size)
    pad = pad // 2
assert isinstance(pad, int) and pad >= 0

padded_image = np.pad(image, pad_width=pad, mode="constant")
i_size = padded_image.shape[0]

result_size = (i_size - f_size + 1) // stride
result = np.zeros((result_size, result_size))

for i, y in enumerate(range(0, i_size - f_size + 1, stride)):
    for j, x in enumerate(range(0, i_size - f_size + 1, stride)):
        result[i, j] = (padded_image[y:y+f_size, x:x+f_size] * filter).sum()

print(result)

[[ 118.  184.  217.  250.  283.  316.  202.]
 [ 288.  420.  456.  492.  528.  564.  348.]
 [ 477.  672.  708.  744.  780.  816.  495.]
 [ 666.  924.  960.  996. 1032. 1068.  642.]
 [ 855. 1176. 1212. 1248. 1284. 1320.  789.]
 [1044. 1428. 1464. 1500. 1536. 1572.  936.]
 [ 490.  628.  643.  658.  673.  688.  374.]]


In [5]:
from typing import Any


from numpy import dtype, floating, ndarray
from numpy._typing._nbit_base import _32Bit


image = np.array(
    [[1, 1, 2, 4],
     [5, 6, 7, 8],
     [3, 2, 1, 0],
     [1, 2, 3, 4]]
).astype(np.float32)

i_size = int(image.shape[0])
f_size = 2
stride = 2
o_size = (i_size - f_size) // stride + 1

out = np.zeros((o_size, o_size))

for i, y in enumerate(range(0, o_size + 1, stride)):
    print(f"{i=}")
    for j, x in enumerate(range(0, o_size + 1, stride)):
        print(f"{j=}")
        slice = image[y:y+f_size, x:x+f_size]
        print(slice)
        out[i, j] = slice.max()

i=0
j=0
[[1. 1.]
 [5. 6.]]
j=1
[[2. 4.]
 [7. 8.]]
i=1
j=0
[[3. 2.]
 [1. 2.]]
j=1
[[1. 0.]
 [3. 4.]]


In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import torchvision.utils as vutils

from src.models.LeNet import get_trained_model, get_train_loader

train_dl = get_train_loader()
model, _, _ = get_trained_model()
model.eval()
images, labels = next(iter(train_dl))
images = images.to(device)
with torch.no_grad():
    logits = model(images)
    softmaxes = F.softmax(logits, dim=1)
    predictions = torch.argmax(logits, dim=1)

images = images.cpu()
softmaxes = softmaxes.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

index_slider = widgets.IntSlider(value=0, min=0, max=len(images) - 1, step=1, description="Index")

def plot_image_and_softmax(idx: int):
    img = images[idx].squeeze(0).numpy()
    probs = softmaxes[idx].numpy()
    pred = predictions[idx].item()
    label = labels[idx].item()

    fig, axs = plt.subplots(1, 2, figsize=(10, 4)) # type: ignore

    # Plot the image
    axs[0].imshow(img, cmap="gray")
    axs[0].axis("off")
    axs[0].set_title(f"True Label: {label} | Predicted: {pred}")

    # Plot the softmax bar chart
    axs[1].bar(np.arange(10), probs)
    axs[1].set_xticks(np.arange(10))
    axs[1].set_ylim([0, 1])
    axs[1].set_title("Softmax Probabilities")
    axs[1].set_ylabel("Confidence")
    axs[1].set_xlabel("Class")

    plt.tight_layout()
    plt.show() # type: ignore

widgets.interact(plot_image_and_softmax, idx=index_slider)

Training: 100%|██████████| 938/938 [00:27<00:00, 33.56it/s]


RuntimeError: Input type (MPSFloatType) and weight type (torch.FloatTensor) should be the same